# 01 — Frame-based AlexNet → PCA → Ridge (improved organizer baseline)

**Same idea as the organizers:** sample eight frames from each 3-second video, obtain frozen AlexNet embeddings, fit PCA and Ridge to reconstruct the EEG, then rank by correlation. **No raw-pixel features, audio model, custom neural network, or transductive assignment.**

Improvements: preserve frame order as a candidate (instead of always averaging frames); use pooled early **and** late AlexNet features; restrict regression to the EEG response window; use train-only channel normalization; select the frame representation/window using development subjects' validation labels only.

Run once with folders `stimuli_train/`, `stimuli_test/`, `trn/`, `tst/`, and `tst_stim/` alongside this notebook.

In [1]:
from pathlib import Path
import json
import numpy as np
import torch
import torchvision
from frame_baseline import (
    TEST_IDS, WEIGHTS_NAME, extract_video_features, load_train_eeg,
    load_test_eeg, select_features, fit_video_projection,
    project_video, fit_eeg_encoder, similarity, ranked_video_ids,
    validate_rankings, save_encoder,
)

ROOT = Path('.').resolve()     # change if your dataset is in a different directory
CACHE = ROOT / 'cache'
MODELS = ROOT / 'models'
CACHE.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

print('Data root:', ROOT)
print('AlexNet weights:', WEIGHTS_NAME)
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)

Data root: D:\eeg
AlexNet weights: torchvision.models.AlexNet_Weights.IMAGENET1K_V1
torch: 2.14.1+cpu | torchvision: 0.29.1+cpu


## 1. Extract **frame-level** CNN representations once

The 8 frames are individually encoded with frozen ImageNet AlexNet. Early and late layer maps are pooled to 2 × 2; each frame becomes a **1280-dimensional neural feature** (not a pixel vector). The same video cache is reusable for all six subjects. Only the training-video cache is ever used to fit PCA and Ridge.

In [2]:
train_video_ids = np.arange(1, 1001, dtype=np.int64)
train_frames = extract_video_features(
    ROOT / 'stimuli_train', train_video_ids, CACHE / 'alexnet_train_frames.npz'
)
test_frames = extract_video_features(
    ROOT / 'stimuli_test', TEST_IDS, CACHE / 'alexnet_test_frames.npz'
)
print('Training frame features:', train_frames.shape)  # (1000, 8, 1280)
print('Candidate frame features:', test_frames.shape)    # (102, 8, 1280)

Extracted 100/1000 from D:\eeg\stimuli_train
Extracted 200/1000 from D:\eeg\stimuli_train
Extracted 300/1000 from D:\eeg\stimuli_train
Extracted 400/1000 from D:\eeg\stimuli_train
Extracted 500/1000 from D:\eeg\stimuli_train
Extracted 600/1000 from D:\eeg\stimuli_train
Extracted 700/1000 from D:\eeg\stimuli_train
Extracted 800/1000 from D:\eeg\stimuli_train
Extracted 900/1000 from D:\eeg\stimuli_train
Extracted 1000/1000 from D:\eeg\stimuli_train
Extracted 100/102 from D:\eeg\stimuli_test
Training frame features: (1000, 8, 1280)
Candidate frame features: (102, 8, 1280)


## 2. Validate two minimal extensions of the baseline

- `mean`: the organizer's average-of-frames idea (with pooled early+late CNN features).
- `frames`: keep the 8 AlexNet frame embeddings **in their temporal order**.

The two EEG windows below are measured in seconds after video onset. Both PCA and Ridge are fitted **only on `trn/sub-XX_trn.npz`**. Development `tst`/`tst_stim` for subjects 01–04 are used solely for choosing the configuration and reporting top-1/top-5 accuracy.

In [3]:
MODES = ('mean', 'frames')
WINDOWS = ((0.05, 0.35), (0.05, 0.70))
PCA_COMPONENTS = 64

results = []
for sub in range(1, 5):   # 01–04: validation and model selection ONLY
    train_ids, train_eeg = load_train_eeg(ROOT, sub)
    X = select_features(train_frames, train_ids)
    val_eeg = load_test_eeg(ROOT, sub)     # original, UNSORTED test row order
    val_labels = np.load(ROOT / 'tst_stim' / f'sub-{sub:02d}_tst_stim.npy')

    for mode in MODES:
        projection, X_train = fit_video_projection(X, mode, PCA_COMPONENTS)
        for window in WINDOWS:
            encoder = fit_eeg_encoder(X_train, train_eeg, window, projection)
            scores = similarity(encoder, val_eeg, test_frames)
            ranked = ranked_video_ids(scores)
            metrics = validate_rankings(ranked, val_labels)
            result = dict(subject=sub, mode=mode, window=window, **metrics)
            results.append(result)
            print(f"sub-{sub:02d} {mode:6s} {window}: "
                  f"top1={metrics['top1']:.3f}, top5={metrics['top5']:.3f}, "
                  f"median-rank={metrics['median_rank']:.1f}")

sub-01 mean   (0.05, 0.35): top1=0.049, top5=0.206, median-rank=18.0
sub-01 mean   (0.05, 0.7): top1=0.049, top5=0.216, median-rank=14.0
sub-01 frames (0.05, 0.35): top1=0.049, top5=0.235, median-rank=18.0
sub-01 frames (0.05, 0.7): top1=0.029, top5=0.245, median-rank=14.0
sub-02 mean   (0.05, 0.35): top1=0.049, top5=0.157, median-rank=20.0
sub-02 mean   (0.05, 0.7): top1=0.029, top5=0.186, median-rank=16.0
sub-02 frames (0.05, 0.35): top1=0.078, top5=0.196, median-rank=18.0
sub-02 frames (0.05, 0.7): top1=0.049, top5=0.235, median-rank=16.0
sub-03 mean   (0.05, 0.35): top1=0.020, top5=0.206, median-rank=22.0
sub-03 mean   (0.05, 0.7): top1=0.069, top5=0.167, median-rank=20.5
sub-03 frames (0.05, 0.35): top1=0.039, top5=0.186, median-rank=22.0
sub-03 frames (0.05, 0.7): top1=0.078, top5=0.167, median-rank=23.5
sub-04 mean   (0.05, 0.35): top1=0.029, top5=0.147, median-rank=31.0
sub-04 mean   (0.05, 0.7): top1=0.039, top5=0.147, median-rank=23.5
sub-04 frames (0.05, 0.35): top1=0.049, t

## 3. Select one configuration and fit on **all six subjects' training EEG**

Selecting a single configuration from average development-subject performance avoids tuning independently to hidden test labels. The hackathon emphasizes top-1 and top-5, so the simple selection score gives top-1 greater weight.

In [4]:
summary = []
for mode in MODES:
    for window in WINDOWS:
        subset = [r for r in results if r['mode'] == mode and r['window'] == window]
        t1 = np.mean([r['top1'] for r in subset])
        t5 = np.mean([r['top5'] for r in subset])
        summary.append((0.7*t1 + 0.3*t5, t1, t5, mode, window))

for selection_score, top1, top5, mode, window in sorted(summary, reverse=True):
    print(f'mode={mode:6s} window={window} top1={top1:.3f} '
          f'top5={top5:.3f} selection={selection_score:.3f}')

_, dev_top1, dev_top5, BEST_MODE, BEST_WINDOW = max(summary)
print('SELECTED:', BEST_MODE, BEST_WINDOW)

for sub in range(1, 7):
    train_ids, train_eeg = load_train_eeg(ROOT, sub)
    X = select_features(train_frames, train_ids)
    projection, X_train = fit_video_projection(X, BEST_MODE, PCA_COMPONENTS)
    encoder = fit_eeg_encoder(X_train, train_eeg, BEST_WINDOW, projection)
    path = MODELS / f'encoder_sub-{sub:02d}.joblib'
    save_encoder(encoder, path)
    print(f'Saved {path}, training recordings={len(train_ids)}, '
          f'ridge alpha={encoder["ridge"].alpha_:.2g}')

metadata = {
    'feature': WEIGHTS_NAME,
    'torch_version': torch.__version__,
    'torchvision_version': torchvision.__version__,
    'n_frames': 8, 'pca_components': PCA_COMPONENTS,
    'feature_mode': BEST_MODE, 'eeg_window_seconds': BEST_WINDOW,
    'mean_development_top1': float(dev_top1),
    'mean_development_top5': float(dev_top5),
    'test_used_for_training': False,
}
(MODELS / 'selection.json').write_text(json.dumps(metadata, indent=2))
print('Training finished. Run notebook 02 to create the two submissions.')

mode=frames window=(0.05, 0.7) top1=0.054 top5=0.201 selection=0.098
mode=frames window=(0.05, 0.35) top1=0.054 top5=0.181 selection=0.092
mode=mean   window=(0.05, 0.7) top1=0.047 top5=0.179 selection=0.086
mode=mean   window=(0.05, 0.35) top1=0.037 top5=0.179 selection=0.079
SELECTED: frames (0.05, 0.7)
Saved D:\eeg\models\encoder_sub-01.joblib, training recordings=968, ridge alpha=1e+03
Saved D:\eeg\models\encoder_sub-02.joblib, training recordings=972, ridge alpha=1e+03
Saved D:\eeg\models\encoder_sub-03.joblib, training recordings=961, ridge alpha=1e+03
Saved D:\eeg\models\encoder_sub-04.joblib, training recordings=947, ridge alpha=1e+03
Saved D:\eeg\models\encoder_sub-05.joblib, training recordings=952, ridge alpha=1e+03
Saved D:\eeg\models\encoder_sub-06.joblib, training recordings=850, ridge alpha=4.6e+03
Training finished. Run notebook 02 to create the two submissions.
